# LangGraph 测试Notebook
---
目标：验证 LangGraph `StateGraph` 基础流转，建立 `GlobalState` 定义与各层节点骨架。

该Notebook是另外一个的改写，只保留真正实现了的逻辑。

In [1]:
# ── 依赖安装 ──────────────────────────────────────────────────────────────────
%pip install -q python-dotenv
%pip install -qU langchain-core langchain-ollama
%pip install -qU langgraph
%pip install -qU langchain-anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

# ── 导入与环境初始化 ───────────────────────────────────────────────────────────
import os,sys
# Jupyter notebooks don't define __file__, so locate the project root by
# searching upward for CLAUDE.md (the project marker file).
def _find_project_root(marker: str = "CLAUDE.md") -> str:
    path = os.path.abspath("")
    for _ in range(6):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(f"Project root not found (searched for '{marker}')")

_root = _find_project_root()
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Project root:", _root)
import getpass
import logging
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_anthropic import ChatAnthropic

from langgraph.graph import StateGraph, START, END

from langchain.agents import create_agent
from pydantic import BaseModel, Field
from SkiLib.metatools.informative import get_tools as get_info_tools
from SkiLib.registry import SkillRegistry

# 从项目根目录 .env 加载 API Key 与追踪配置
load_dotenv()

# LangSmith 追踪（可选 — 在 .env 中设置 LANGSMITH_TRACING=true 启用）
if os.getenv("LANGSMITH_TRACING", "false").lower() == "true":
    if not os.getenv("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API Key: ")
    if not os.getenv("LANGSMITH_PROJECT"):
        os.environ["LANGSMITH_PROJECT"] = getpass.getpass("Enter LangSmith Project name: ")

logging.basicConfig(level=logging.ERROR, force=True)
print("Environment loaded. LangSmith tracing:", os.getenv("LANGSMITH_TRACING", "false"))

Project root: c:\Users\fengy\RoboSkiAgent


c:\Users\fengy\RoboSkiAgent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment loaded. LangSmith tracing: true


In [3]:
LLM_TYPE = "claude"  # 可选 "claude" 或 "ollama"（本地）
# ── LLM 配置 ──────────────────────────────────────────────────────────────────
# 使用本地 Ollama 时修改 OLLAMA_MODEL_ID 切换模型
OLLAMA_MODEL_ID = os.getenv("OLLAMA_MODEL_ID", "qwen3:latest")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

if LLM_TYPE == "claude":
    llm = ChatAnthropic(
        model="claude-sonnet-4-6"
    )
else:
    llm = ChatOllama(
        model=OLLAMA_MODEL_ID,
        base_url=OLLAMA_BASE_URL,
        temperature=0,
    )

# 快速连通性检查
try:
    _ping = llm.invoke("Reply with one word: ready")
    print("LLM reachable:", _ping.content.strip()[:80])
except Exception as e:
    print(f"[WARN] LLM not reachable — nodes using llm will fail. ({e})")

LLM reachable: ready


## GlobalState
与 `CLAUDE.md` 中 `GlobalState` 规范保持一致。  
`robot_state` 此处用 `dict` 代替，生产版本从 `SkiLib.base` 导入 `RobotState`。

In [4]:
# ── GlobalState 定义 ──────────────────────────────────────────────────────────
from typing import TypedDict, Annotated, Optional, Literal
import operator


class GlobalState(TypedDict):
    # Layer-1：规划层输出
    todo_list: list[dict]           # [{task_id, type, skill/description, params}, ...]

    # Layer-2：执行上下文
    current_task: dict              # 执行槽：{} = 空闲，{...} = 执行中或失败保留

    # 机器人运行时快照（此处用 dict 代替，生产类型：SkiLib.base.RobotState）
    robot_state: dict

    # 控制标志
    halt_flag: bool                 # True = 所有 R-skill 执行被锁定
    halt_reason: Optional[str]      # "TASK_FAILURE" | "MANUAL_TASK" | None

    # Executor 写入；含 needs_hilp 字段供 Context Flush 决策
    last_result: Optional[dict]

    # 内部路由字段：human_intervention 写入，after_human_intervention 读取
    _hi_action: Optional[str]

    # 审计日志，由 Context Flush 写入；Annotated list 避免键覆盖
    execution_log: Annotated[list[str], operator.add]

    # LangGraph 消息总线
    messages: Annotated[list[BaseMessage], operator.add]


print("GlobalState keys:", list(GlobalState.__annotations__.keys()))

GlobalState keys: ['todo_list', 'current_task', 'robot_state', 'halt_flag', 'halt_reason', 'last_result', '_hi_action', 'execution_log', 'messages']


In [5]:
# ── 节点：Supervisor（新版，create_agent, create_react_agent已经废弃）────────────────────────────────
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from SkiLib.metatools.informative import get_tools as get_info_tools
from SkiLib.registry import SkillRegistry


# ── SupervisorOutput Schema ───────────────────────────────────────────────────
# available_skills 不在此 schema 中 — 由代码从 SkillRegistry 注入，LLM 不填写。
class SupervisorOutput(BaseModel):
    """Fact sheet produced after knowledge saturation. Symbol-only, no coordinates."""
    task_intent_original: str = Field(description="Verbatim user instruction")
    
    task_intent: str = Field(
        description="Rewritten instruction using exact RoboDK symbol names"
    )
    
    scene: dict = Field(
        description="Keys: targets (list[str]), objects (list[str]), tools (list[str])"
    )
    
    # available_skills: This will be injected by code, not filled by LLM. Keys: skill name, docstring.
    
    extra_info: str = Field(
        default="",
        description="Unresolvable ambiguities or free-text observations"
    )


def _get_available_skills() -> dict:
    """Pure-code: read skill signatures from SkillRegistry. No LLM involved."""
    registry = SkillRegistry.instance()
    if not registry:
        return {}
    return {
        name: registry.get_skill(name).execute.__doc__ or ""
        for name in registry.list_skills()
    }


# ── System Prompt ─────────────────────────────────────────────────────────────
def _build_supervisor_prompt() -> str:
    skills_text = "\n".join(
        f"  - {name}: {doc.strip()}"
        for name, doc in _get_available_skills().items()
    ) or "  (none registered)"

    return f"""\
You are the Supervisor agent in an industrial robot assembly system.

Your ONLY job: gather scene facts and produce a structured SupervisorOutput summary.
Do NOT plan tasks, choose skills, or compute coordinates.

Rules:
- Call tools to query the RoboDK scene until you have enough information.
- Rewrite the instruction in `task_intent` using exact RoboDK symbol names (e.g. "Part_A_1", not "part A").
- All `scene` fields must use exact names returned by query tools — no invented names.
- Never include coordinates, joint angles, or numeric poses anywhere in your output.
- Record unresolvable ambiguities in `extra_info` instead of guessing.

Available skills (for your reference only — do not invent new ones):
{skills_text}
"""


# ── Lazy singleton ────────────────────────────────────────────────────────────
_supervisor_agent = None

def _get_supervisor_agent():
    global _supervisor_agent
    if _supervisor_agent is None:
        _supervisor_agent = create_agent(
            model=llm,
            tools=get_info_tools(),
            response_format=SupervisorOutput,
            system_prompt=_build_supervisor_prompt(),
        )
        print("[supervisor] Agent built. Skills:", list(_get_available_skills().keys()))
    return _supervisor_agent


# ── Node function ─────────────────────────────────────────────────────────────
def supervisor(state: GlobalState) -> dict:
    result = _get_supervisor_agent().invoke({"messages": state["messages"]})

    summary: SupervisorOutput | None = result.get("structured_response")
    if summary is None:
        raw = state["messages"][-1].content
        raw_str = raw if isinstance(raw, str) else str(raw)
        summary = SupervisorOutput(
            task_intent_original=raw_str,
            task_intent=raw_str,
            scene={},
            extra_info="response_format unsupported — stub fallback",
        )


    output = summary.model_dump()
    # Inject available_skills from registry (pure code, not LLM output)
    output["available_skills"] = _get_available_skills()

    return {"messages": [AIMessage(content=str(output))]}


print("Supervisor cell loaded.")

Supervisor cell loaded.


In [ ]:
# ── Supervisor 单元测试 ───────────────────────────────────────────────────────
# 前提：RoboDK 已启动
from SkiLib.robotcontext import RobotContext
from SkiLib.registry import SkillRegistry
import json
# 1. 拉起 RoboDK API
context = RobotContext()
RDK    = context.RDK
robot  = context.robot
skill_registry = SkillRegistry.instance()
print("[init] RobotContext OK, skills:", skill_registry.list_skills())

# 2. 重置 supervisor 单例（使其用已初始化的 SkillRegistry 重建）
_supervisor_agent = None

# 3. 构造最小初始状态
_test_state = {
    "messages":      [HumanMessage(content="把 Part_A_1 放到 Place Part A")],
    "todo_list":     [], "current_task": {}, "robot_state": {},
    "halt_flag":     False, "halt_reason": None,
    "last_result":   None, "_hi_action":  None, "execution_log": [],
}

# 4. 运行
print("\n[test] Running supervisor...")
_updates = supervisor(_test_state)

# 5. 解析输出
_out = _updates["messages"][-1].content

[init] RobotContext OK, skills: ['DummySkill', 'PickAndPlace']

[test] Running supervisor...
[supervisor] Agent built. Skills: ['DummySkill', 'PickAndPlace']


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [ ]:
# ── 节点：Planner ─────────────────────────────────────────────────────────────
# 职责：通过强制结构化输出生成 todo_list JSON 任务队列。
# 支持 AutoTask（自动执行）和 ManualTask（触发 human_intervention）混排。
# TODO：替换为真实 LLM 结构化输出 + Pydantic schema 校验 + retry

# ── Task Models ───────────────────────────────────────────────────────────────
# todo_list 中每个任务的 Pydantic schema。
# AutoTask   → Executor 执行，skill + params 必填。
# ManualTask → Dispatcher 直接路由到 human_intervention，不经 Executor。
from pydantic import BaseModel, Field
from typing import Literal

class AutoTask(BaseModel):
    task_id: str
    type: Literal["auto"] = "auto"
    skill: str
    params: dict = Field(default_factory=dict)

class ManualTask(BaseModel):
    task_id: str
    type: Literal["manual"] = "manual"
    description: str
    
# Union alias — 供 Planner 构造、Dispatcher 路由时做类型收窄
Task = AutoTask | ManualTask


class PlannerOutput(BaseModel):
    todo_list: list[Task]



print("Task models defined:", [AutoTask.__name__, ManualTask.__name__])

def _build_planner_prompt() -> str:
    template = """
    You are the planner of a symobolic robot assembly system. Your ONLY job is to generate a
    task plan as a JSON list of tasks, each being either an AutoTask (with a skill used and params) or 
    a ManualTask (with a description for human intervention). 
    Do NOT include any information about coordinates, robot states, or execution details — 
    only the high-level task structure.
    
    Rules:
    - Each AutoTask must specify a skill from the available skills and follow its parameter schema.
    - Always insert ManualTask whenever the task cannot be done by skills in the list, do not invent new skills.
    - Always require human intervention if any ambiguity or uncertainty cannot be resolved, including but not limited to:
      - Missing or ambiguous skill parameters
    
    Here are some example of vaild tasks:
    - {"task_id": "t1", "type": "manual", "description": "Flip and screw the assembly before placing part C."}
    - {"task_id": "t2", "type": "auto", "skill": "PickAndPlace", "params": {"item": Part_A_1,
        "pick_approach": "App Pick Part A",
        "pick_target": "Pick Part A",
        "place_approach": "App Place Part A",
        "place_target": "Place Part A"
    }}
    """

def planner(state: GlobalState) -> dict:
    print("[planner] Generating task plan...")
    strucutured_llm = llm.with_structured_output(PlannerOutput,method="json_schema")
    
    return {
        "todo_list": todo_dicts,
        "messages":  [AIMessage(content=f"[Planner] {len(todo)} tasks queued ({manual_count} manual)")],
    }